In [ ]:

import sys
import pandas as pd
import numpy as np
import tensorflow as tf

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Pandas: 2.2.3
NumPy: 2.1.3
TensorFlow: 2.21.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:

import pandas as pd

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

df = pd.read_csv(url, sep="\t", header=None, names=["label", "text"])

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nLabel counts:")
print(df["label"].value_counts())


Dataset shape: (5572, 2)

First 5 rows:


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



Label counts:
label
ham     4825
spam     747
Name: count, dtype: int64


In [ ]:

import pandas as pd

url = "https://huggingface.co/datasets/zefang-liu/phishing-email-dataset/resolve/main/Phishing_Email.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nLabel counts:")
for col in df.columns:
    if "type" in col.lower() or "label" in col.lower() or "class" in col.lower():
        print(df[col].value_counts(dropna=False))



Dataset shape: (18650, 3)

Column names:
['Unnamed: 0', 'Email Text', 'Email Type']

First 5 rows:


,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email



Label counts:
Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64


In [ ]:

# Step 5: Clean and inspect the email dataset

df = df[["Email Text", "Email Type"]].copy()

df = df.dropna(subset=["Email Text", "Email Type"])
df["Email Text"] = df["Email Text"].astype(str).str.strip()
df["Email Type"] = df["Email Type"].astype(str).str.strip()

df = df[df["Email Text"] != ""]
df = df.drop_duplicates(subset=["Email Text"])

print("Dataset shape after cleaning:", df.shape)
print("\nClass counts:")
print(df["Email Type"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

print("\nExample safe email:")
display(df[df["Email Type"] == "Safe Email"]["Email Text"].head(1))

print("\nExample phishing email:")
display(df[df["Email Type"] == "Phishing Email"]["Email Text"].head(1))


Dataset shape after cleaning: (17522, 2)

Class counts:
Email Type
Safe Email        10978
Phishing Email     6544
Name: count, dtype: int64

Missing values:
Email Text    0
Email Type    0
dtype: int64

Example safe email:


,Email Text
0,"re : 6 . 1100 , disc : uniformitarianism , re ..."



Example phishing email:


,Email Text
3,Hello I am your hot lil horny toy.\n I am t...


In [ ]:

from sklearn.model_selection import train_test_split

# Convert text labels to numbers:
# Safe Email = 0, Phishing Email = 1
label_map = {
    "Safe Email": 0,
    "Phishing Email": 1
}

df["label"] = df["Email Type"].map(label_map)

X = df["Email Text"].astype(str)
y = df["label"]

# 80% training, 20% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Split temporary data equally into validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training emails:", len(X_train))
print("Validation emails:", len(X_val))
print("Testing emails:", len(X_test))

print("\nTraining class counts:")
print(y_train.value_counts().sort_index())

print("\nLabel mapping:", label_map)


Training emails: 14017
Validation emails: 1752
Testing emails: 1753

Training class counts:
label
0    8782
1    5235
Name: count, dtype: int64

Label mapping: {'Safe Email': 0, 'Phishing Email': 1}


In [ ]:

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Settings
MAX_WORDS = 20000
MAX_LEN = 250

# Fit tokenizer ONLY on training emails
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

# Convert emails into sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Make all sequences the same length
X_train_pad = pad_sequences(
    X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post"
)
X_val_pad = pad_sequences(
    X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post"
)
X_test_pad = pad_sequences(
    X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post"
)

# Convert labels to NumPy arrays
y_train_np = y_train.to_numpy(dtype=np.int32)
y_val_np = y_val.to_numpy(dtype=np.int32)
y_test_np = y_test.to_numpy(dtype=np.int32)

print("Training shape:", X_train_pad.shape)
print("Validation shape:", X_val_pad.shape)
print("Testing shape:", X_test_pad.shape)
print("Vocabulary size:", min(len(tokenizer.word_index) + 1, MAX_WORDS))
print("\nExample token sequence:")
print(X_train_pad[0][:30])


Training shape: (14017, 250)
Validation shape: (1752, 250)
Testing shape: (1753, 250)
Vocabulary size: 20000

Example token sequence:
[ 1164    95     1     2  2293   250   276    13  2489  2293    12     6
   924   677     6   311   265   311  2293   316     6 10071  3475   702
    12     1   113    39   642    17]


In [ ]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Embedding, SpatialDropout1D,
    Bidirectional, LSTM, Dropout, Dense
)

tf.keras.utils.set_random_seed(42)

model = Sequential([
    Input(shape=(MAX_LEN,), dtype="int32"),

    Embedding(
        input_dim=MAX_WORDS,
        output_dim=64,
        mask_zero=True
    ),

    SpatialDropout1D(0.2),

    Bidirectional(LSTM(32)),

    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 250, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 250, 64)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,306,945 (4.99 MB)

 Trainable params: 1,306,945 (4.99 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        "best_email_phishing_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

history = model.fit(
    X_train_pad,
    y_train_np,
    validation_data=(X_val_pad, y_val_np),
    epochs=8,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")
print("Best validation accuracy:",
      round(max(history.history["val_accuracy"]) * 100, 2), "%")


Epoch 1/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7986 - loss: 0.3920
Epoch 1: val_loss improved from None to 0.04264, saving model to best_email_phishing_model.keras

Epoch 1: finished saving model to best_email_phishing_model.keras
220/220 ━━━━━━━━━━━━━━━━━━━━ 12s 27ms/step - accuracy: 0.9086 - loss: 0.2060 - val_accuracy: 0.9857 - val_loss: 0.0426
Epoch 2/8
219/220 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9874 - loss: 0.0402
Epoch 2: val_loss did not improve from 0.04264
220/220 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.9906 - loss: 0.0312 - val_accuracy: 0.9874 - val_loss: 0.0482
Epoch 3/8
219/220 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9963 - loss: 0.0117
Epoch 3: val_loss improved from 0.04264 to 0.03842, saving model to best_email_phishing_model.keras

Epoch 3: finished saving model to best_email_phishing_model.keras
220/220 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.9969 - loss: 0.0106 - val_accuracy: 0.9897 - val_loss: 0.0384
Epoch 4/8

In [ ]:

import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Load the saved best model
best_model = tf.keras.models.load_model(
    "best_email_phishing_model.keras"
)

# Evaluate on the separate test set
test_loss, test_accuracy = best_model.evaluate(
    X_test_pad, y_test_np, verbose=0
)

# Convert prediction scores into class labels
probabilities = best_model.predict(X_test_pad, verbose=0).ravel()
y_pred = (probabilities >= 0.5).astype(int)

print(f"Test accuracy: {test_accuracy * 100:.2f}%")
print(f"Test loss: {test_loss:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test_np,
    y_pred,
    labels=[0, 1],
    target_names=["Safe Email", "Phishing Email"],
    digits=4,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test_np, y_pred, labels=[0, 1]))


Test accuracy: 98.23%
Test loss: 0.0698

Classification report:
                precision    recall  f1-score   support

    Safe Email     0.9899    0.9818    0.9858      1098
Phishing Email     0.9699    0.9832    0.9765       655

      accuracy                         0.9823      1753
     macro avg     0.9799    0.9825    0.9812      1753
  weighted avg     0.9824    0.9823    0.9823      1753

Confusion matrix:
[[1078   20]
 [  11  644]]


In [ ]:

import json
import os
import pickle
import tensorflow as tf

os.makedirs("email_phishing_artifacts", exist_ok=True)

# Save the best trained model
best_model.save("email_phishing_artifacts/email_phishing_model.keras")

# Save tokenizer
with open("email_phishing_artifacts/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save settings required for prediction
metadata = {
    "project": "Email Phishing Detector",
    "architecture": "Bidirectional LSTM",
    "max_words": MAX_WORDS,
    "max_len": MAX_LEN,
    "padding": "post",
    "truncating": "post",
    "labels": {
        "0": "Safe Email",
        "1": "Phishing Email"
    },
    "threshold": 0.5,
    "test_accuracy": round(float(test_accuracy), 6)
}

with open("email_phishing_artifacts/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Saved files:")
print(os.listdir("email_phishing_artifacts"))


Saved files:
['metadata.json', 'tokenizer.pkl', 'email_phishing_model.keras']


In [ ]:

import json
import os

output_dir = "email_phishing_artifacts"

# Export the exact vocabulary and text-processing settings
tokenizer_data = {
    "word_index": tokenizer.word_index,
    "num_words": MAX_WORDS,
    "max_len": MAX_LEN,
    "oov_token": tokenizer.oov_token,
    "filters": tokenizer.filters,
    "lower": tokenizer.lower,
    "split": tokenizer.split,
    "padding": "post",
    "truncating": "post"
}

with open(
    os.path.join(output_dir, "tokenizer.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(tokenizer_data, f)

print("Tokenizer JSON saved successfully.")
print("Vocabulary entries:", len(tokenizer.word_index))
print("Files now available:")
print(sorted(os.listdir(output_dir)))


Tokenizer JSON saved successfully.
Vocabulary entries: 153296
Files now available:
['email_phishing_model.keras', 'metadata.json', 'tokenizer.json', 'tokenizer.pkl']


In [ ]:

%pip install tensorflowjs


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 94.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of wheel to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of tensorflow-decision-forests to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of tensorflow-decision-forests to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of tensorflow to determine 

In [ ]:

import tensorflow as tf
import tensorflowjs as tfjs
import packaging

print("TensorFlow:", tf.__version__)
print("TensorFlow.js:", tfjs.__version__ if hasattr(tfjs, "__version__") else "Imported")
print("Packaging:", packaging.__version__)


/usr/local/lib/python3.13/dist-packages/tensorflowjs/read_weights.py:28: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  np.uint8, np.uint16, np.object, np.bool]


AttributeError: module 'numpy' has no attribute 'object'.
`np.object` was a deprecated alias for the builtin `object`. To avoid this error in existing code, use `object` by itself. Doing this will not modify any behavior and is safe. 
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

In [ ]:

import numpy as np

# Compatibility aliases for the older converter
if "object" not in np.__dict__:
    np.object = object

if "bool" not in np.__dict__:
    np.bool = bool

import tensorflow as tf
import tensorflowjs as tfjs
import os

print("TensorFlow:", tf.__version__)
print("TensorFlow.js converter imported successfully!")

# Check whether your saved model files survived the restart
folder = "/content/email_phishing_artifacts"
print("\nSaved artifacts folder exists:", os.path.isdir(folder))

if os.path.isdir(folder):
    print("Files:", sorted(os.listdir(folder)))


AttributeError: module 'tensorflow.compat.v1' has no attribute 'estimator'

In [ ]:

import os

print("Current directory:", os.getcwd())
print("\nAvailable files and folders:")

for name in os.listdir("/content"):
    print(name)


Current directory: /content

Available files and folders:
.config
email_phishing_artifacts
best_email_phishing_model.keras
sample_data


In [ ]:

import os

folder = "/content/email_phishing_artifacts"

print("Artifact files:")
for name in sorted(os.listdir(folder)):
    path = os.path.join(folder, name)
    print(f"{name} — {os.path.getsize(path) / 1024:.1f} KB")

print("\nStandalone best model exists:",
      os.path.exists("/content/best_email_phishing_model.keras"))


Artifact files:
email_phishing_model.keras — 15364.5 KB
metadata.json — 0.3 KB
tokenizer.json — 3101.4 KB
tokenizer.pkl — 6763.8 KB

Standalone best model exists: True


In [ ]:

%pip install --upgrade "tensorflowjs==4.22.0"


  Using cached tensorflowjs-4.22.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached tensorflow_decision_forests-1.8.1.tar.gz (15.2 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached packaging-23.2-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is looking at multiple versions of tensorflow-decision-forests to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of tensorflow-decision-forests to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
ERROR: Canno

In [ ]:

%pip show tensorflow tensorflowjs tensorflow-decision-forests


Name: tensorflow
Version: 2.21.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: absl-py, astunparse, flatbuffers, gast, google_pasta, grpcio, h5py, keras, libclang, ml_dtypes, numpy, opt_einsum, packaging, protobuf, requests, setuptools, six, termcolor, typing_extensions, wrapt
Required-by: dopamine_rl, tensorflow-text, tensorflowjs, tf_keras, ydf_tf
---
Name: tensorflowjs
Version: 3.18.0
Summary: 
Home-page: https://js.tensorflow.org/
Author: Google LLC
Author-email: opensource@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: packaging, six, tensorflow, tensorflow-hub
Required-by: 


In [ ]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/email_phishing_artifacts_backup",
    "zip",
    "/content/email_phishing_artifacts"
)

files.download("/content/email_phishing_artifacts_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

files_to_check = [
    "/content/email_phishing_artifacts/email_phishing_model.keras",
    "/content/email_phishing_artifacts/tokenizer.json",
    "/content/email_phishing_artifacts/metadata.json"
]

for file in files_to_check:
    print(os.path.basename(file), "EXISTS:", os.path.exists(file))

email_phishing_model.keras EXISTS: True
tokenizer.json EXISTS: True
metadata.json EXISTS: True


In [ ]:
import tensorflow as tf
import os

model_path = "/content/email_phishing_artifacts/email_phishing_model.keras"

model = tf.keras.models.load_model(model_path)

print("Model loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("TensorFlow version:", tf.__version__)

Model loaded successfully!
Input shape: (None, 250)
Output shape: (None, 1)
TensorFlow version: 2.21.0


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

test_email = [
    "Urgent! Your bank account is suspended. Click this link immediately to verify your password."
]

tokenizer_path = "/content/email_phishing_artifacts/tokenizer.pkl"
import pickle

with open(tokenizer_path, "rb") as f:
    tokenizer = pickle.load(f)

sequence = tokenizer.texts_to_sequences(test_email)
padded = pad_sequences(
    sequence,
    maxlen=250,
    padding="post",
    truncating="post"
)

probability = float(model.predict(padded, verbose=0)[0][0])

print("Phishing probability:", round(probability * 100, 2), "%")
print("Prediction:", "Phishing Email" if probability >= 0.5 else "Safe Email")

Phishing probability: 99.99 %
Prediction: Phishing Email


In [ ]:
import json
import os

artifact_dir = "/content/email_phishing_artifacts"

print("Files ready for the website:")
for filename in sorted(os.listdir(artifact_dir)):
    path = os.path.join(artifact_dir, filename)
    print(f"{filename} — {os.path.getsize(path) / (1024 * 1024):.2f} MB")

Files ready for the website:
email_phishing_model.keras — 15.00 MB
metadata.json — 0.00 MB
tokenizer.json — 3.03 MB
tokenizer.pkl — 6.61 MB


In [ ]:

import os

folder = "/content/email_phishing_artifacts"

print("Folder exists:", os.path.exists(folder))

if os.path.exists(folder):
    for name in os.listdir(folder):
        path = os.path.join(folder, name)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{name} — {size_mb:.2f} MB")


Folder exists: True
tokenizer.json — 3.03 MB
metadata.json — 0.00 MB
tokenizer.pkl — 6.61 MB
email_phishing_model.keras — 15.00 MB


In [ ]:

import tensorflow as tf
import os

model_path = "/content/email_phishing_artifacts/email_phishing_model.keras"

print("TensorFlow version:", tf.__version__)
print("Model file exists:", os.path.exists(model_path))

model = tf.keras.models.load_model(model_path)

print("Model loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)


TensorFlow version: 2.21.0
Model file exists: True
Model loaded successfully!
Input shape: (None, 250)
Output shape: (None, 1)


In [ ]:

import importlib.metadata as md

for package in ["tensorflow", "tensorflowjs", "keras", "numpy"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")


tensorflow: 2.21.0
tensorflowjs: 3.18.0
keras: 3.13.2
numpy: 2.1.3


In [ ]:

import os
import tensorflow as tf

export_path = "/content/email_phishing_savedmodel"

if not os.path.exists(export_path):
    model.export(export_path)

print("SavedModel export exists:", os.path.exists(export_path))
print("Export contents:", os.listdir(export_path))


Saved artifact at '/content/email_phishing_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 250), dtype=tf.int32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140054460561936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570035216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570037328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570035792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140054458174160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570036560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570036176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140054458187408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140054458187600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140054458187024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1400515

In [ ]:

import shutil
import os

source = "/content/email_phishing_savedmodel"
zip_path = "/content/email_phishing_savedmodel"

shutil.make_archive(zip_path, "zip", source)

print("ZIP created:", os.path.exists(zip_path + ".zip"))
print("ZIP size:", round(os.path.getsize(zip_path + ".zip") / (1024 * 1024), 2), "MB")


ZIP created: True
ZIP size: 9.26 MB


In [ ]:

from google.colab import files

files.download("/content/email_phishing_savedmodel.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import sys
import importlib.metadata as md

print("Python:", sys.version)

for package in ["tensorflow", "tensorflowjs", "keras", "numpy"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("SavedModel folder exists:",
      __import__("os").path.exists("/content/email_phishing_savedmodel"))


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
tensorflow: 2.21.0
tensorflowjs: 3.18.0
keras: 3.13.2
numpy: 2.1.3
SavedModel folder exists: True


In [ ]:

!tensorflowjs_converter \
  --input_format=tf_saved_model \
  --output_format=tfjs_graph_model \
  /content/email_phishing_savedmodel \
  /content/email_phishing_tfjs


/usr/local/lib/python3.13/dist-packages/tensorflowjs/read_weights.py:28: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  np.uint8, np.uint16, np.object, np.bool]
Traceback (most recent call last):
  File "/usr/local/bin/tensorflowjs_converter", line 3, in <module>
    from tensorflowjs.converters.converter import pip_main
  File "/usr/local/lib/python3.13/dist-packages/tensorflowjs/__init__.py", line 21, in <module>
    from tensorflowjs import converters
  File "/usr/local/lib/python3.13/dist-packages/tensorflowjs/converters/__init__.py", line 21, in <module>
    from tensorflowjs.converters.converter import convert
  File "/usr/local/lib/python3.13/dist-packages/tensorflowjs/converters/converter.py", line 35, in <module>
    from tensorflowjs.converters import keras_h5_conversion as conversion
  File "/usr/local/lib/python3.13/dist-packages/tensorflowjs/converters/keras_h5_conversion.py", line 33, in <module>
    from tensorflowjs import 

In [ ]:

import sys
import numpy as np

# Temporary compatibility aliases for the older converter
np.object = object
np.bool = bool

sys.argv = [
    "tensorflowjs_converter",
    "--input_format=tf_saved_model",
    "--output_format=tfjs_graph_model",
    "/content/email_phishing_savedmodel",
    "/content/email_phishing_tfjs"
]

from tensorflowjs.converters.converter import pip_main
pip_main()


AttributeError: module 'tensorflow.compat.v1' has no attribute 'estimator'

In [ ]:

import os
import tensorflow as tf

print("SavedModel files:")
print(os.listdir("/content/email_phishing_savedmodel"))

loaded_model = tf.saved_model.load("/content/email_phishing_savedmodel")
print("\nModel loaded successfully.")
print("Available functions:", list(loaded_model.signatures.keys()))

print("\nOriginal model backup exists:",
      os.path.exists("/content/email_phishing_artifacts/email_phishing_model.keras"))

print("Tokenizer exists:",
      os.path.exists("/content/email_phishing_artifacts/tokenizer.json"))


SavedModel files:
['saved_model.pb', 'variables', 'assets', 'fingerprint.pb']

Model loaded successfully.
Available functions: ['serve', 'serving_default']

Original model backup exists: True
Tokenizer exists: True


In [ ]:

import json
import numpy as np
import tensorflow as tf

# Load the saved tokenizer configuration
with open("/content/email_phishing_artifacts/tokenizer.json", "r") as f:
    tok = json.load(f)

word_index = tok["word_index"]
max_words = tok["num_words"]
max_len = tok["max_len"]
oov_token = tok["oov_token"]

# Prepare text using the saved tokenizer settings
def prepare_email(text):
    if tok["lower"]:
        text = text.lower()

    filters = tok["filters"]
    table = str.maketrans({char: tok["split"] for char in filters})
    words = text.translate(table).split(tok["split"])

    sequence = []
    for word in words:
        if not word:
            continue
        index = word_index.get(word)
        if index is None or (max_words and index >= max_words):
            index = word_index.get(oov_token, 1)
        sequence.append(index)

    sequence = sequence[:max_len]
    sequence += [0] * (max_len - len(sequence))

    return np.array([sequence], dtype=np.int32)

# Run the actual trained model
model = tf.saved_model.load("/content/email_phishing_savedmodel")
predict_fn = model.signatures["serve"]

sample_email = (
    "Urgent! Your bank account is suspended. "
    "Click this link immediately to verify your password."
)

input_tensor = tf.constant(prepare_email(sample_email))
result = predict_fn(input_layer=input_tensor)
probability = float(list(result.values())[0].numpy().reshape(-1)[0])

print("Phishing probability:", round(probability * 100, 2), "%")
print("Prediction:", "Phishing Email" if probability >= 0.5 else "Safe Email")


Phishing probability: 99.99 %
Prediction: Phishing Email


In [ ]:

import os
import shutil

base = "/content/email_phishing_artifacts"
package = "/content/email_phishing_deployment"
zip_path = "/content/email_phishing_deployment.zip"

# Prepare a clean deployment folder
if os.path.exists(package):
    shutil.rmtree(package)

os.makedirs(package)

# Copy the working SavedModel
shutil.copytree(
    "/content/email_phishing_savedmodel",
    os.path.join(package, "model")
)

# Copy tokenizer and model metadata
shutil.copy2(
    os.path.join(base, "tokenizer.json"),
    os.path.join(package, "tokenizer.json")
)

shutil.copy2(
    os.path.join(base, "metadata.json"),
    os.path.join(package, "metadata.json")
)

# Create ZIP
if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    "/content/email_phishing_deployment",
    "zip",
    package
)

print("Deployment ZIP created successfully.")
print("File size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")
print("Files:", os.listdir(package))


Deployment ZIP created successfully.
File size: 10.45 MB
Files: ['tokenizer.json', 'metadata.json', 'model']


In [ ]:

from google.colab import files
files.download("/content/email_phishing_deployment.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model(
    "/content/email_phishing_savedmodel"
)

try:
    tflite_model = converter.convert()

    output_path = "/content/email_phishing_model.tflite"

    with open(output_path, "wb") as f:
        f.write(tflite_model)

    print("Conversion successful!")
    print("Model size:", round(os.path.getsize(output_path) / (1024 * 1024), 2), "MB")

except Exception as e:
    print("Conversion failed:")
    print(type(e).__name__, str(e)[:1500])


Conversion failed:
ConverterError Could not translate MLIR to FlatBuffer.<unknown>:0: error: loc(callsite(fused["CudnnRNNV3:", "sequential_1/bidirectional_1/backward_lstm_1/CudnnRNNV3@__inference___call___1783"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper___call___1838"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]))): 'tf.CudnnRNNV3' op is neither a custom op nor a flex op
<unknown>:0: note: loc(callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper___call___1838"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): called from
<unknown>:0: note: loc(fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]): called from
<unknown>:0: note: loc(callsite(fused["CudnnRNNV3:", "sequential_1/bidirectional_1/backward_lstm_1/CudnnRNNV3@__inference___call___1783"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signat

In [ ]:

import os
import tensorflow as tf

keras_path = "/content/email_phishing_artifacts/email_phishing_model.keras"
output_path = "/content/email_phishing_model.tflite"

try:
    original_model = tf.keras.models.load_model(keras_path)

    converter = tf.lite.TFLiteConverter.from_keras_model(original_model)
    tflite_model = converter.convert()

    with open(output_path, "wb") as f:
        f.write(tflite_model)

    print("SUCCESS: Keras-to-TFLite conversion completed.")
    print("Model size:", round(os.path.getsize(output_path) / (1024 * 1024), 2), "MB")

except Exception as e:
    print("Conversion failed:", type(e).__name__)
    print(str(e)[:2000])


Saved artifact at '/tmp/tmpb5ow30ao'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 250), dtype=tf.int32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140051570037136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570035792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570043280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570040016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570035216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570036368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570034832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570039632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570042128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570042896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140051570034064: TensorSp

In [ ]:
import os

model_path = "/content/email_phishing_artifacts/email_phishing_model.keras"

print("Original model exists:", os.path.exists(model_path))
if os.path.exists(model_path):
    print("Model size (MB):", round(os.path.getsize(model_path) / (1024 * 1024), 2))

Original model exists: False


In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for name in files:
        if (
            "email_phishing_model.keras" in name
            or "email_phishing_artifacts_backup.zip" in name
        ):
            print(os.path.join(root, name))

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for name in files:
        if (
            "email_phishing_model.keras" in name
            or "email_phishing_artifacts_backup.zip" in name
        ):
            print(os.path.join(root, name))

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving email_phishing_model.keras to email_phishing_model.keras


In [ ]:
import tensorflow as tf
from tensorflow import keras
import os

model_path = "/content/email_phishing_model.keras"

print("Model exists:", os.path.exists(model_path))

old_model = keras.models.load_model(model_path, compile=False)
old_model.summary()

Model exists: True


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 250, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 250, 64)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,306,945 (4.99 MB)

 Trainable params: 1,306,945 (4.99 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import os
import shutil
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Create the same architecture with CPU-compatible LSTM
cpu_model = keras.Sequential([
    keras.Input(shape=(250,), dtype="int32"),
    layers.Embedding(input_dim=20000, output_dim=64, mask_zero=True),
    layers.SpatialDropout1D(0.2),
    layers.Bidirectional(
        layers.LSTM(32, use_cudnn=False)
    ),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])

# 2. Build and transfer the existing trained weights
cpu_model(tf.zeros((1, 250), dtype=tf.int32))
cpu_model.set_weights(old_model.get_weights())

print("SUCCESS: Trained weights transferred!")
print("Original parameters:", old_model.count_params())
print("New model parameters:", cpu_model.count_params())

# 3. Test CPU inference
with tf.device("/CPU:0"):
    test_input = tf.zeros((1, 250), dtype=tf.int32)
    test_prediction = cpu_model(test_input, training=False)

print("CPU test prediction:", test_prediction.numpy())

# 4. Export CPU-compatible SavedModel
output_dir = "/content/email_phishing_cpu_savedmodel"
shutil.rmtree(output_dir, ignore_errors=True)

@tf.function(
    input_signature=[
        tf.TensorSpec(
            shape=[None, 250],
            dtype=tf.int32,
            name="input_layer"
        )
    ]
)
def serve(input_layer):
    return {
        "output_0": cpu_model(input_layer, training=False)
    }

tf.saved_model.save(
    cpu_model,
    output_dir,
    signatures={
        "serve": serve,
        "serving_default": serve
    }
)

# 5. Verify the exported model
loaded_model = tf.saved_model.load(output_dir)
loaded_predict = loaded_model.signatures["serve"]

with tf.device("/CPU:0"):
    result = loaded_predict(input_layer=test_input)

print("Exported model test:", result["output_0"].numpy())
print("Available signatures:", list(loaded_model.signatures.keys()))

# 6. Package the new model for download
zip_path = shutil.make_archive(
    "/content/email_phishing_cpu_savedmodel",
    "zip",
    output_dir
)

print("ZIP created:", zip_path)
print("ZIP size (MB):", round(os.path.getsize(zip_path) / (1024 * 1024), 2))

SUCCESS: Trained weights transferred!
Original parameters: 1306945
New model parameters: 1306945
CPU test prediction: [[0.51776975]]
Exported model test: [[0.51776975]]
Available signatures: ['serve', 'serving_default']
ZIP created: /content/email_phishing_cpu_savedmodel.zip
ZIP size (MB): 4.65


In [ ]:
from google.colab import files

files.download("/content/email_phishing_cpu_savedmodel.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>